In [ ]:
import os
import json
import math
import time
import zipfile
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

warnings.filterwarnings("ignore")

CONFIG = {
    "input_pairs_file": "stage2_regime_switching_input.csv",
    "spread_panel_file": "cusip_day_spread_panel_cleaned.csv",
    "spread_column": "daily_spread",
    "date_column": "trade_date",

    "target_pairs_limit": None,   # use None for all rows in input_pairs_file
    "min_obs_for_regime_model": 250,

    # MarkovRegression setup on ΔECT_t = a_s + λ_s * ECT_{t-1} + ε_t,s
    "k_regimes": 2,
    "trend": "c",
    "switching_trend": True,
    "switching_exog": True,
    "switching_variance": True,

    # Fitting controls
    "fit_method": "bfgs",
    "fit_maxiter": 200,
    "fit_em_iter": 10,
    "fit_search_reps": 8,
    "fit_search_iter": 10,
    "fit_search_scale": 1.0,
    "fit_disp": False,

    # Post-fit interpretation / acceptance flags
    "min_regime_occupancy": 0.10,
    "max_transition_persistence": 0.995,
    "min_variance_ratio_for_distinct": 1.25,
    "min_phi_gap_for_distinct": 0.05,

    # Balanced top table
    "top_n_balanced": 300,
    "top_balanced_issuer_cap": 4,
    "top_by_sector_max_per_combo": 8,
    "top_by_sector_issuer_cap": 2,

    # Runtime / outputs
    "checkpoint_every": 50,
    "write_checkpoint_files": True,
    "output_dir": "stage2_regime_switching_outputs",
    "zip_basename": "stage2_regime_switching_model_outputs",
    "zip_outputs": True,
    "auto_download_zip_if_colab": True,
    "random_seed": 42,
}

np.random.seed(CONFIG["random_seed"])


def is_running_in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


def maybe_download_in_colab(path_str: str):
    if CONFIG["auto_download_zip_if_colab"] and is_running_in_colab():
        from google.colab import files  # type: ignore
        files.download(path_str)


def safe_half_life_from_phi(phi):
    if pd.isna(phi):
        return np.nan
    if phi <= 0 or phi >= 1:
        return np.nan
    return math.log(2) / (-math.log(phi))


def build_spread_wide(spread_panel: pd.DataFrame) -> pd.DataFrame:
    wide = (
        spread_panel
        .pivot_table(index=CONFIG["date_column"], columns="cusip_id", values=CONFIG["spread_column"], aggfunc="first")
        .sort_index()
    )
    return wide


def get_param(params, key):
    try:
        return float(params[key])
    except Exception:
        return np.nan


def extract_transition_probs(res):
    try:
        P = np.asarray(res.regime_transition)
        if P.ndim == 3:
            P = P[:, :, 0]
        p00 = float(P[0, 0]); p01 = float(P[1, 0]) if P.shape==(2,2) else np.nan
        # res.regime_transition uses rows as destination, cols as source in example
        # translate to conventional source->dest probabilities
        # source 0: to0=P[0,0], to1=P[1,0]; source1: to0=P[0,1], to1=P[1,1]
        p00 = float(P[0, 0]); p10 = float(P[1, 0]); p01 = float(P[0, 1]); p11 = float(P[1, 1])
        return p00, p01, p10, p11
    except Exception:
        return (np.nan, np.nan, np.nan, np.nan)


def reconstruct_pair_ect(pair_row, spread_wide: pd.DataFrame) -> pd.DataFrame:
    c1 = pair_row["cusip_1"]
    c2 = pair_row["cusip_2"]
    if c1 not in spread_wide.columns or c2 not in spread_wide.columns:
        return pd.DataFrame()
    temp = spread_wide[[c1, c2]].dropna().copy()
    if temp.empty:
        return pd.DataFrame()
    beta1 = pair_row["beta1_normalized"]
    beta2 = pair_row["beta2_normalized"]
    const = pair_row.get("coint_const", 0.0)
    temp = temp.rename(columns={c1: "y1", c2: "y2"}).reset_index()
    temp["ect"] = beta1 * temp["y1"] + beta2 * temp["y2"] + const
    temp["dect"] = temp["ect"].diff()
    temp["ect_lag"] = temp["ect"].shift(1)
    temp = temp.dropna(subset=["dect", "ect_lag"]).reset_index(drop=True)
    return temp


def fit_single_pair(pair_row, spread_wide: pd.DataFrame) -> dict:
    base = {
        "cusip_1": pair_row["cusip_1"],
        "cusip_2": pair_row["cusip_2"],
        "company_symbol_1": pair_row.get("company_symbol_1", np.nan),
        "company_symbol_2": pair_row.get("company_symbol_2", np.nan),
        "sector_1": pair_row.get("sector_1", np.nan),
        "sector_2": pair_row.get("sector_2", np.nan),
        "sector_combo": pair_row.get("sector_combo", np.nan),
        "stage2_regime_score": pair_row.get("stage2_regime_score", np.nan),
        "selection_stage": pair_row.get("selection_stage", np.nan),
        "regime_input_rank": pair_row.get("regime_input_rank", np.nan),
        "trace_margin_95": pair_row.get("trace_margin_95", np.nan),
        "spread_change_corr_shortlist": pair_row.get("spread_change_corr_shortlist", np.nan),
        "avg_remaining_maturity_diff": pair_row.get("avg_remaining_maturity_diff", np.nan),
        "half_life_days_stage1": pair_row.get("half_life_days", np.nan),
        "ect_adf_pvalue_stage1": pair_row.get("ect_adf_pvalue", np.nan),
    }
    try:
        pair_df = reconstruct_pair_ect(pair_row, spread_wide)
        n_obs = int(len(pair_df))
        base["n_obs_regime"] = n_obs
        if n_obs < CONFIG["min_obs_for_regime_model"]:
            base.update({
                "status": "skipped",
                "skip_reason": "too_few_regime_obs"
            })
            return base

        endog = pair_df["dect"].astype(float)
        exog = pair_df[["ect_lag"]].astype(float)

        model = MarkovRegression(
            endog=endog,
            k_regimes=CONFIG["k_regimes"],
            trend=CONFIG["trend"],
            exog=exog,
            switching_trend=CONFIG["switching_trend"],
            switching_exog=CONFIG["switching_exog"],
            switching_variance=CONFIG["switching_variance"],
        )
        res = model.fit(
            method=CONFIG["fit_method"],
            maxiter=CONFIG["fit_maxiter"],
            em_iter=CONFIG["fit_em_iter"],
            search_reps=CONFIG["fit_search_reps"],
            search_iter=CONFIG["fit_search_iter"],
            search_scale=CONFIG["fit_search_scale"],
            disp=CONFIG["fit_disp"],
        )

        params = res.params
        bse = getattr(res, "bse", pd.Series(index=params.index, data=np.nan))
        pvalues = getattr(res, "pvalues", pd.Series(index=params.index, data=np.nan))
        converged = bool(getattr(res, "mle_retvals", {}).get("converged", True))

        intercept0 = get_param(params, "const[0]")
        intercept1 = get_param(params, "const[1]")
        lam0 = get_param(params, "x1[0]")
        lam1 = get_param(params, "x1[1]")
        sigma20 = get_param(params, "sigma2[0]")
        sigma21 = get_param(params, "sigma2[1]")

        implied_phi0 = 1.0 + lam0 if pd.notna(lam0) else np.nan
        implied_phi1 = 1.0 + lam1 if pd.notna(lam1) else np.nan
        hl0 = safe_half_life_from_phi(implied_phi0)
        hl1 = safe_half_life_from_phi(implied_phi1)

        p00, p01, p10, p11 = extract_transition_probs(res)
        dur0 = np.nan if pd.isna(p00) or p00 >= 1 else 1.0 / (1.0 - p00)
        dur1 = np.nan if pd.isna(p11) or p11 >= 1 else 1.0 / (1.0 - p11)

        smoothed = pd.DataFrame(res.smoothed_marginal_probabilities)
        occ0 = float(smoothed.iloc[:, 0].mean())
        occ1 = float(smoothed.iloc[:, 1].mean())
        maxprob0 = float(smoothed.iloc[:, 0].max())
        maxprob1 = float(smoothed.iloc[:, 1].max())

        variance_ratio = np.nan
        if pd.notna(sigma20) and pd.notna(sigma21) and min(sigma20, sigma21) > 0:
            variance_ratio = max(sigma20, sigma21) / min(sigma20, sigma21)

        phi_gap = np.nan
        if pd.notna(implied_phi0) and pd.notna(implied_phi1):
            phi_gap = abs(implied_phi0 - implied_phi1)

        min_occupancy = np.nanmin([occ0, occ1])
        max_persistence = np.nanmax([p00, p11])

        distinct_regimes = (
            (pd.notna(variance_ratio) and variance_ratio >= CONFIG["min_variance_ratio_for_distinct"]) or
            (pd.notna(phi_gap) and phi_gap >= CONFIG["min_phi_gap_for_distinct"])
        )
        nondegenerate_transitions = pd.notna(max_persistence) and (max_persistence <= CONFIG["max_transition_persistence"])
        sufficient_occupancy = pd.notna(min_occupancy) and (min_occupancy >= CONFIG["min_regime_occupancy"])

        regime_useful = bool(converged and distinct_regimes and nondegenerate_transitions and sufficient_occupancy)

        # label lower-vol regime as normal if possible
        normal_regime = np.nan
        stress_regime = np.nan
        if pd.notna(sigma20) and pd.notna(sigma21):
            normal_regime = int(0 if sigma20 <= sigma21 else 1)
            stress_regime = int(1 - normal_regime)

        base.update({
            "status": "fitted",
            "skip_reason": np.nan,
            "converged": converged,
            "llf_regime": float(res.llf),
            "aic_regime": float(res.aic),
            "bic_regime": float(res.bic),
            "hqic_regime": float(res.hqic) if hasattr(res, "hqic") else np.nan,

            "intercept_regime0": intercept0,
            "intercept_regime1": intercept1,
            "lambda_regime0": lam0,
            "lambda_regime1": lam1,
            "lambda_pvalue_regime0": get_param(pvalues, "x1[0]"),
            "lambda_pvalue_regime1": get_param(pvalues, "x1[1]"),
            "intercept_pvalue_regime0": get_param(pvalues, "const[0]"),
            "intercept_pvalue_regime1": get_param(pvalues, "const[1]"),

            "implied_phi_regime0": implied_phi0,
            "implied_phi_regime1": implied_phi1,
            "implied_half_life_regime0": hl0,
            "implied_half_life_regime1": hl1,
            "sigma2_regime0": sigma20,
            "sigma2_regime1": sigma21,
            "variance_ratio": variance_ratio,
            "phi_gap": phi_gap,

            "p00": p00,
            "p01": p01,
            "p10": p10,
            "p11": p11,
            "expected_duration_regime0": dur0,
            "expected_duration_regime1": dur1,

            "occupancy_regime0": occ0,
            "occupancy_regime1": occ1,
            "min_regime_occupancy": min_occupancy,
            "max_smoothed_prob_regime0": maxprob0,
            "max_smoothed_prob_regime1": maxprob1,

            "normal_regime_by_variance": normal_regime,
            "stress_regime_by_variance": stress_regime,

            "passes_regime_occupancy": bool(sufficient_occupancy),
            "passes_regime_transition_nondegenerate": bool(nondegenerate_transitions),
            "passes_regime_distinctness": bool(distinct_regimes),
            "regime_useful": regime_useful,
        })

        # optional time series for accepted subset later
        base["_pair_df"] = pair_df
        base["_smoothed"] = smoothed

        return base

    except Exception as e:
        base.update({
            "status": "failed",
            "skip_reason": f"{type(e).__name__}: {str(e)[:200]}",
        })
        return base


def write_checkpoint(df: pd.DataFrame, out_dir: Path, suffix: str):
    path = out_dir / suffix
    df.to_csv(path, index=False)


def add_rankings(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    fitted = out["status"].eq("fitted")
    # Lower rank score is better because percentiles use ascending=True for some metrics
    if fitted.any():
        tmp = out.loc[fitted].copy()
        # composite ranking, lower is better
        for col, asc, newcol in [
            ("aic_regime", True, "rank_aic"),
            ("variance_ratio", False, "rank_variance_ratio"),
            ("phi_gap", False, "rank_phi_gap"),
            ("min_regime_occupancy", False, "rank_occupancy"),
            ("trace_margin_95", False, "rank_trace_margin"),
            ("stage2_regime_score", False, "rank_stage2_input"),
        ]:
            vals = tmp[col]
            tmp[newcol] = vals.rank(pct=True, ascending=asc)
        tmp["regime_rank_score"] = (
            0.20 * tmp["rank_aic"] +
            0.15 * tmp["rank_variance_ratio"] +
            0.15 * tmp["rank_phi_gap"] +
            0.15 * tmp["rank_occupancy"] +
            0.15 * tmp["rank_trace_margin"] +
            0.20 * tmp["rank_stage2_input"]
        )
        out = out.merge(tmp[["cusip_1","cusip_2","regime_rank_score","rank_aic","rank_variance_ratio","rank_phi_gap","rank_occupancy","rank_trace_margin","rank_stage2_input"]],
                        on=["cusip_1","cusip_2"], how="left")
    return out


def select_balanced(df: pd.DataFrame, top_n: int, issuer_cap: int, per_combo_cap: int | None = None) -> pd.DataFrame:
    usable = df[df["regime_useful"] == True].sort_values(["regime_rank_score", "aic_regime"]).copy()
    issuer_counts = {}
    combo_issuer_counts = {}
    selected_rows = []

    for _, row in usable.iterrows():
        issuers = [row["company_symbol_1"], row["company_symbol_2"]]
        combo = row["sector_combo"]
        ok = True
        for iss in issuers:
            if issuer_counts.get(iss, 0) >= issuer_cap:
                ok = False
                break
            if per_combo_cap is not None and combo_issuer_counts.get((combo, iss), 0) >= per_combo_cap:
                ok = False
                break
        if not ok:
            continue
        selected_rows.append(row)
        for iss in issuers:
            issuer_counts[iss] = issuer_counts.get(iss, 0) + 1
            if per_combo_cap is not None:
                combo_issuer_counts[(combo, iss)] = combo_issuer_counts.get((combo, iss), 0) + 1
        if len(selected_rows) >= top_n:
            break

    return pd.DataFrame(selected_rows)




In [ ]:
def main():
    t0 = time.time()
    out_dir = Path(CONFIG["output_dir"])
    out_dir.mkdir(exist_ok=True, parents=True)

    pairs = pd.read_csv(CONFIG["input_pairs_file"])
    if CONFIG["target_pairs_limit"] is not None:
        pairs = pairs.head(int(CONFIG["target_pairs_limit"])).copy()

    spread = pd.read_csv(CONFIG["spread_panel_file"], parse_dates=[CONFIG["date_column"]])
    spread_wide = build_spread_wide(spread)

    with open(out_dir / "stage2_regime_switching_run_config.json", "w") as f:
        json.dump(CONFIG, f, indent=2)

    results = []
    ect_series_records = []
    failed = []

    for i, (_, row) in enumerate(pairs.iterrows(), start=1):
        res = fit_single_pair(row, spread_wide)
        pair_df = res.pop("_pair_df", None)
        smoothed = res.pop("_smoothed", None)
        results.append(res)

        if res.get("status") == "failed":
            failed.append(res)

        # save ect + probabilities for useful fits only
        if pair_df is not None and smoothed is not None and res.get("regime_useful", False):
            temp = pair_df.copy()
            temp["prob_regime0"] = smoothed.iloc[:, 0].values
            temp["prob_regime1"] = smoothed.iloc[:, 1].values
            temp["cusip_1"] = res["cusip_1"]
            temp["cusip_2"] = res["cusip_2"]
            temp["company_symbol_1"] = res["company_symbol_1"]
            temp["company_symbol_2"] = res["company_symbol_2"]
            ect_series_records.append(temp)

        if CONFIG["write_checkpoint_files"] and (i % CONFIG["checkpoint_every"] == 0):
            chk = pd.DataFrame(results)
            write_checkpoint(chk, out_dir, "stage2_regime_results_checkpoint.csv")

    res_df = pd.DataFrame(results)
    res_df = add_rankings(res_df)

    useful_df = res_df[res_df["regime_useful"] == True].copy()
    failed_df = res_df[res_df["status"].isin(["failed", "skipped"])].copy()

    balanced_top = select_balanced(
        res_df,
        top_n=CONFIG["top_n_balanced"],
        issuer_cap=CONFIG["top_balanced_issuer_cap"],
        per_combo_cap=None
    )
    by_sector_list = []
    for combo, g in useful_df.sort_values(["regime_rank_score", "aic_regime"]).groupby("sector_combo"):
        take = select_balanced(g, top_n=CONFIG["top_by_sector_max_per_combo"], issuer_cap=CONFIG["top_by_sector_issuer_cap"], per_combo_cap=CONFIG["top_by_sector_issuer_cap"])
        if not take.empty:
            by_sector_list.append(take)
    by_sector = pd.concat(by_sector_list, ignore_index=True) if by_sector_list else pd.DataFrame()

    ect_series = pd.concat(ect_series_records, ignore_index=True) if ect_series_records else pd.DataFrame()

    # summaries
    runtime_sec = time.time() - t0
    report = pd.DataFrame([{
        "input_pairs": int(len(pairs)),
        "fitted_pairs": int((res_df["status"] == "fitted").sum()),
        "useful_pairs": int((res_df["regime_useful"] == True).sum()),
        "failed_pairs": int((res_df["status"] == "failed").sum()),
        "skipped_pairs": int((res_df["status"] == "skipped").sum()),
        "useful_share_of_input": float((res_df["regime_useful"] == True).mean()) if len(res_df) else np.nan,
        "median_aic_useful": float(useful_df["aic_regime"].median()) if not useful_df.empty else np.nan,
        "median_variance_ratio_useful": float(useful_df["variance_ratio"].median()) if not useful_df.empty else np.nan,
        "median_phi_gap_useful": float(useful_df["phi_gap"].median()) if not useful_df.empty else np.nan,
        "median_min_occupancy_useful": float(useful_df["min_regime_occupancy"].median()) if not useful_df.empty else np.nan,
        "median_persistence_useful": float(np.nanmedian(np.maximum(useful_df["p00"], useful_df["p11"]))) if not useful_df.empty else np.nan,
        "runtime_seconds": runtime_sec,
        "runtime_minutes": runtime_sec / 60.0,
    }])

    breakdown = pd.DataFrame([
        {"filter_name": "passes_regime_occupancy", "pass_count": int((res_df["passes_regime_occupancy"] == True).sum()), "pass_rate_of_fitted": float((res_df.loc[res_df["status"]=="fitted", "passes_regime_occupancy"] == True).mean()) if (res_df["status"]=="fitted").any() else np.nan},
        {"filter_name": "passes_regime_transition_nondegenerate", "pass_count": int((res_df["passes_regime_transition_nondegenerate"] == True).sum()), "pass_rate_of_fitted": float((res_df.loc[res_df["status"]=="fitted", "passes_regime_transition_nondegenerate"] == True).mean()) if (res_df["status"]=="fitted").any() else np.nan},
        {"filter_name": "passes_regime_distinctness", "pass_count": int((res_df["passes_regime_distinctness"] == True).sum()), "pass_rate_of_fitted": float((res_df.loc[res_df["status"]=="fitted", "passes_regime_distinctness"] == True).mean()) if (res_df["status"]=="fitted").any() else np.nan},
        {"filter_name": "regime_useful", "pass_count": int((res_df["regime_useful"] == True).sum()), "pass_rate_of_input": float((res_df["regime_useful"] == True).mean()) if len(res_df) else np.nan},
    ])

    issuer_usage = pd.concat([
        useful_df[["company_symbol_1"]].rename(columns={"company_symbol_1": "issuer"}),
        useful_df[["company_symbol_2"]].rename(columns={"company_symbol_2": "issuer"})
    ], ignore_index=True).value_counts().reset_index(name="useful_pair_appearances") if not useful_df.empty else pd.DataFrame(columns=["issuer", "useful_pair_appearances"])

    sector_counts = useful_df["sector_combo"].value_counts().reset_index()
    if not sector_counts.empty:
        sector_counts.columns = ["sector_combo", "useful_pair_count"]

    # write outputs
    res_df.to_csv(out_dir / "stage2_regime_results_all.csv", index=False)
    useful_df.to_csv(out_dir / "stage2_regime_results_useful.csv", index=False)
    failed_df.to_csv(out_dir / "stage2_regime_results_failed_or_skipped.csv", index=False)
    balanced_top.to_csv(out_dir / "stage2_top_regime_pairs_balanced.csv", index=False)
    by_sector.to_csv(out_dir / "stage2_top_regime_pairs_by_sector_combo.csv", index=False)
    report.to_csv(out_dir / "stage2_regime_report.csv", index=False)
    breakdown.to_csv(out_dir / "stage2_regime_filter_breakdown.csv", index=False)
    issuer_usage.to_csv(out_dir / "stage2_regime_issuer_usage.csv", index=False)
    sector_counts.to_csv(out_dir / "stage2_regime_sector_combo_counts.csv", index=False)
    if not ect_series.empty:
        ect_series.to_csv(out_dir / "stage2_regime_ect_probabilities_useful.csv", index=False)

    readme = """Stage 2 regime-switching output guide
====================================

Files produced by this script:

1. stage2_regime_results_all.csv
   One row per input pair with full regime-switching diagnostics.

2. stage2_regime_results_useful.csv
   Pairs whose regime models are considered useful:
   - converged
   - both regimes have meaningful occupancy
   - transition probabilities are not degenerate
   - regimes are economically distinct by variance ratio and/or phi gap

3. stage2_regime_results_failed_or_skipped.csv
   Failed fits or skipped pairs.

4. stage2_top_regime_pairs_balanced.csv
   Balanced top regime-switching pairs.

5. stage2_top_regime_pairs_by_sector_combo.csv
   Top useful regime pairs by sector combination.

6. stage2_regime_report.csv
   Summary counts and medians for the useful regime set.

7. stage2_regime_filter_breakdown.csv
   Pass rates for each regime-quality filter.

8. stage2_regime_issuer_usage.csv
   Issuer frequency in the useful regime set.

9. stage2_regime_sector_combo_counts.csv
   Sector combination counts in the useful regime set.

10. stage2_regime_ect_probabilities_useful.csv (optional)
    Useful-pair ECT time series with smoothed regime probabilities.

Model specification:
ΔECT_t = a_{S_t} + λ_{S_t} ECT_{t-1} + ε_{t,S_t}

Interpreting coefficients:
- λ_s < 0 implies mean reversion in regime s
- implied level-AR coefficient is phi_s = 1 + λ_s
- implied half-life is computed from phi_s when 0 < phi_s < 1
"""
    with open(out_dir / "README_stage2_regime_outputs.txt", "w") as f:
        f.write(readme)

    if CONFIG["zip_outputs"]:
        zip_path = str(out_dir) + ".zip"
        if os.path.exists(zip_path):
            os.remove(zip_path)
        shutil.make_archive(str(out_dir), "zip", root_dir=str(out_dir))
        maybe_download_in_colab(zip_path)

    print(report.to_string(index=False))
    print(f"Outputs written to: {out_dir.resolve()}")

In [ ]:
main()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 input_pairs  fitted_pairs  useful_pairs  failed_pairs  skipped_pairs  useful_share_of_input  median_aic_useful  median_variance_ratio_useful  median_phi_gap_useful  median_min_occupancy_useful  median_persistence_useful  runtime_seconds  runtime_minutes
        1000           827           442           173              0                  0.442       -1832.051176                      3.399658               0.291392                     0.297367                   0.932714       647.547653        10.792461
Outputs written to: /content/stage2_regime_switching_outputs


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import zipfile
from typing import Dict, Any

CONFIG: Dict[str, Any] = {
    "input_useful_file": "stage2_regime_results_useful.csv",
    "variance_shift_threshold": 1.5,
    "phi_gap_threshold": 0.10,
    "high_quality_min_occupancy": 0.15,
    "high_quality_min_variance_ratio": 1.5,
    "high_quality_min_phi_gap": 0.10,
    "high_quality_max_abs_lambda": 1.0,
    "presentation_target_n": 100,
    "presentation_issuer_cap": 4,
    "presentation_sector_combo_cap": 4,
    "output_dir": "stage2_regime_post_analysis_outputs",
    "zip_basename": "stage2_regime_post_analysis_outputs",
    "zip_outputs": True,
    "auto_download_zip_if_colab": True,
}
CONFIG

{'input_useful_file': 'stage2_regime_results_useful.csv',
 'variance_shift_threshold': 1.5,
 'phi_gap_threshold': 0.1,
 'high_quality_min_occupancy': 0.15,
 'high_quality_min_variance_ratio': 1.5,
 'high_quality_min_phi_gap': 0.1,
 'high_quality_max_abs_lambda': 1.0,
 'presentation_target_n': 100,
 'presentation_issuer_cap': 4,
 'presentation_sector_combo_cap': 4,
 'output_dir': 'stage2_regime_post_analysis_outputs',
 'zip_basename': 'stage2_regime_post_analysis_outputs',
 'zip_outputs': True,
 'auto_download_zip_if_colab': True}

In [ ]:
def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def safe_half_life(phi: float):
    if pd.isna(phi):
        return np.nan
    if phi <= 0 or phi >= 1:
        return np.nan
    return np.log(2) / -np.log(phi)

def classify_distinctness(var_ratio: float, phi_gap: float, var_thr: float, phi_thr: float) -> str:
    var_flag = pd.notna(var_ratio) and var_ratio >= var_thr
    phi_flag = pd.notna(phi_gap) and phi_gap >= phi_thr
    if var_flag and phi_flag:
        return "both_variance_and_reversion_shift"
    if var_flag:
        return "variance_shift_dominated"
    if phi_flag:
        return "reversion_shift_dominated"
    return "borderline_useful"

def classify_reversion_pattern(row: pd.Series) -> str:
    lam0 = row.get("lambda_regime0", np.nan)
    lam1 = row.get("lambda_regime1", np.nan)
    phi0 = row.get("implied_phi_regime0", np.nan)
    phi1 = row.get("implied_phi_regime1", np.nan)

    mr0 = pd.notna(lam0) and lam0 < 0 and pd.notna(phi0) and 0 < phi0 < 1
    mr1 = pd.notna(lam1) and lam1 < 0 and pd.notna(phi1) and 0 < phi1 < 1

    if mr0 and mr1:
        return "both_regimes_mean_reverting"
    if mr0 and not mr1:
        return "regime0_only_mean_reverting"
    if mr1 and not mr0:
        return "regime1_only_mean_reverting"
    return "neither_regime_cleanly_mean_reverting"

def classify_stress_driver(row: pd.Series, var_thr: float, phi_thr: float) -> str:
    var_flag = pd.notna(row["variance_ratio"]) and row["variance_ratio"] >= var_thr
    phi_flag = pd.notna(row["phi_gap"]) and row["phi_gap"] >= phi_thr
    if var_flag and phi_flag:
        return "both_variance_and_adjustment"
    if var_flag:
        return "variance_only"
    if phi_flag:
        return "adjustment_only"
    return "weak_distinction"

def build_presentation_subset(df: pd.DataFrame, target_n: int, issuer_cap: int, sector_cap: int) -> pd.DataFrame:
    selected = []
    issuer_counts = {}
    sector_counts = {}
    for _, row in df.sort_values(["post_regime_rank_score", "regime_rank_score"]).iterrows():
        scombo = row["sector_combo"]
        issuers = [row["company_symbol_1"], row["company_symbol_2"]]
        if sector_counts.get(scombo, 0) >= sector_cap:
            continue
        if any(issuer_counts.get(iss, 0) >= issuer_cap for iss in issuers):
            continue
        selected.append(row)
        sector_counts[scombo] = sector_counts.get(scombo, 0) + 1
        for iss in issuers:
            issuer_counts[iss] = issuer_counts.get(iss, 0) + 1
        if len(selected) >= target_n:
            break
    return pd.DataFrame(selected)

def maybe_colab_download(path: str) -> None:
    if not CONFIG["auto_download_zip_if_colab"]:
        return
    try:
        from google.colab import files  # type: ignore
        files.download(path)
    except Exception:
        pass

In [ ]:

df = pd.read_csv(CONFIG["input_useful_file"]).copy()
df["regime_distinction_type"] = df.apply(
    lambda r: classify_distinctness(
        r["variance_ratio"], r["phi_gap"],
        CONFIG["variance_shift_threshold"], CONFIG["phi_gap_threshold"]
    ), axis=1
)
df["reversion_pattern_type"] = df.apply(classify_reversion_pattern, axis=1)
df["stress_driver_type"] = df.apply(
    lambda r: classify_stress_driver(
        r, CONFIG["variance_shift_threshold"], CONFIG["phi_gap_threshold"]
    ), axis=1
)
df["high_quality_useful"] = (
    (df["min_regime_occupancy"] >= CONFIG["high_quality_min_occupancy"]) &
    (df["variance_ratio"] >= CONFIG["high_quality_min_variance_ratio"]) &
    (df["phi_gap"] >= CONFIG["high_quality_min_phi_gap"]) &
    (df["lambda_regime0"].abs() <= CONFIG["high_quality_max_abs_lambda"]) &
    (df["lambda_regime1"].abs() <= CONFIG["high_quality_max_abs_lambda"])
)
df["regime_half_life0_recomputed"] = df["implied_phi_regime0"].apply(safe_half_life)
df["regime_half_life1_recomputed"] = df["implied_phi_regime1"].apply(safe_half_life)
df["half_life_ratio_max_to_min"] = np.where(
    df[["regime_half_life0_recomputed", "regime_half_life1_recomputed"]].min(axis=1) > 0,
    df[["regime_half_life0_recomputed", "regime_half_life1_recomputed"]].max(axis=1) /
    df[["regime_half_life0_recomputed", "regime_half_life1_recomputed"]].min(axis=1),
    np.nan
)

def pct_rank(series: pd.Series, ascending=True):
    return series.rank(pct=True, ascending=ascending)

df["rank_varratio_post"] = pct_rank(df["variance_ratio"], ascending=False)
df["rank_phigap_post"] = pct_rank(df["phi_gap"], ascending=False)
df["rank_occupancy_post"] = pct_rank(df["min_regime_occupancy"], ascending=False)
df["rank_trace_post"] = pct_rank(df["trace_margin_95"], ascending=False)
df["rank_corr_post"] = pct_rank(df["spread_change_corr_shortlist"], ascending=False)
df["rank_maturity_post"] = pct_rank(df["avg_remaining_maturity_diff"], ascending=True)
df["rank_ectp_post"] = pct_rank(df["ect_adf_pvalue_stage1"], ascending=True)

df["post_regime_rank_score"] = (
    0.22 * df["rank_varratio_post"] +
    0.20 * df["rank_phigap_post"] +
    0.14 * df["rank_occupancy_post"] +
    0.14 * df["rank_trace_post"] +
    0.10 * df["rank_corr_post"] +
    0.10 * df["rank_maturity_post"] +
    0.10 * df["rank_ectp_post"]
)
df["post_regime_rank"] = df["post_regime_rank_score"].rank(ascending=False, method="first").astype(int)

presentation_pool = df[df["high_quality_useful"]].sort_values("post_regime_rank_score", ascending=False).copy()
presentation = build_presentation_subset(
    presentation_pool,
    target_n=CONFIG["presentation_target_n"],
    issuer_cap=CONFIG["presentation_issuer_cap"],
    sector_cap=CONFIG["presentation_sector_combo_cap"]
)
df.head()

,cusip_1,cusip_2,company_symbol_1,company_symbol_2,sector_1,sector_2,sector_combo,stage2_regime_score,selection_stage,regime_input_rank,...,half_life_ratio_max_to_min,rank_varratio_post,rank_phigap_post,rank_occupancy_post,rank_trace_post,rank_corr_post,rank_maturity_post,rank_ectp_post,post_regime_rank_score,post_regime_rank
0,00287YBX6,053332AZ5,ABBV,AZO,Health Care,Consumer Discretionary,Consumer Discretionary | Health Care,0.649744,guaranteed_sector_combo,2,...,6.431752,0.590498,0.226244,0.877828,0.022624,0.033937,0.325792,0.090498,0.346244,423
1,00287YBX6,009158BC9,ABBV,APD,Health Care,Materials,Health Care | Materials,0.646756,guaranteed_sector_combo,3,...,1.703600,0.837104,0.852941,0.119910,0.013575,0.038462,0.414027,0.061086,0.424796,336
2,031162DR8,126408HU0,AMGN,CSX,Health Care,Industrials,Health Care | Industrials,0.637525,guaranteed_sector_combo,4,...,1.933823,0.110860,0.837104,0.705882,0.067873,0.018100,0.246606,0.042986,0.330905,428
3,02209SBL6,053332BB7,MO,AZO,Consumer Staples,Consumer Discretionary,Consumer Discretionary | Consumer Staples,0.625934,guaranteed_sector_combo,6,...,2.496412,0.692308,0.608597,0.642534,0.101810,0.049774,0.420814,0.269231,0.452217,295
4,053332BH4,125523CS7,AZO,CI,Consumer Discretionary,Health Care,Consumer Discretionary | Health Care,0.622641,guaranteed_sector_combo,7,...,1.852677,0.966063,0.848416,0.153846,0.153846,0.151584,0.581448,0.122172,0.510814,204


In [ ]:

summary_counts = pd.DataFrame({
    "metric": [
        "useful_pairs_in",
        "high_quality_useful_pairs",
        "presentation_pairs_selected",
        "unique_sector_combos_useful",
        "unique_sector_combos_presentation",
        "unique_issuers_useful",
        "unique_issuers_presentation",
    ],
    "value": [
        int(len(df)),
        int(df["high_quality_useful"].sum()),
        int(len(presentation)),
        int(df["sector_combo"].nunique()),
        int(presentation["sector_combo"].nunique()) if len(presentation) else 0,
        int(pd.unique(pd.concat([df["company_symbol_1"], df["company_symbol_2"]])).size),
        int(pd.unique(pd.concat([presentation["company_symbol_1"], presentation["company_symbol_2"]])).size) if len(presentation) else 0,
    ]
})
type_counts = (
    df.groupby(["regime_distinction_type", "reversion_pattern_type", "stress_driver_type"], dropna=False)
      .size()
      .reset_index(name="n_pairs")
      .sort_values("n_pairs", ascending=False)
)
sector_combo_summary = (
    df.groupby("sector_combo")
      .agg(
          n_pairs=("sector_combo", "size"),
          n_high_quality=("high_quality_useful", "sum"),
          median_variance_ratio=("variance_ratio", "median"),
          median_phi_gap=("phi_gap", "median"),
          median_min_occupancy=("min_regime_occupancy", "median"),
          median_half_life_stage1=("half_life_days_stage1", "median"),
      )
      .reset_index()
      .sort_values(["n_high_quality", "n_pairs"], ascending=False)
)
summary_counts, type_counts.head(10), presentation.head()

(                              metric  value
 0                    useful_pairs_in    442
 1          high_quality_useful_pairs    252
 2        presentation_pairs_selected     96
 3        unique_sector_combos_useful     54
 4  unique_sector_combos_presentation     38
 5              unique_issuers_useful     65
 6        unique_issuers_presentation     59,
              regime_distinction_type       reversion_pattern_type  \
 1  both_variance_and_reversion_shift  both_regimes_mean_reverting   
 7           variance_shift_dominated  both_regimes_mean_reverting   
 4          reversion_shift_dominated  both_regimes_mean_reverting   
 3  both_variance_and_reversion_shift  regime1_only_mean_reverting   
 2  both_variance_and_reversion_shift  regime0_only_mean_reverting   
 0                  borderline_useful  both_regimes_mean_reverting   
 6          reversion_shift_dominated  regime1_only_mean_reverting   
 5          reversion_shift_dominated  regime0_only_mean_reverting   
 
       

In [ ]:

outdir = CONFIG["output_dir"]
ensure_dir(outdir)

comparison = df[[
    "cusip_1","cusip_2","company_symbol_1","company_symbol_2","sector_combo",
    "half_life_days_stage1","implied_half_life_regime0","implied_half_life_regime1",
    "regime_half_life0_recomputed","regime_half_life1_recomputed",
    "variance_ratio","phi_gap","min_regime_occupancy",
    "regime_distinction_type","reversion_pattern_type","stress_driver_type",
    "high_quality_useful","post_regime_rank_score","post_regime_rank"
]].sort_values("post_regime_rank_score", ascending=False)

issuer_usage = pd.concat([
    df[["company_symbol_1"]].rename(columns={"company_symbol_1":"issuer"}),
    df[["company_symbol_2"]].rename(columns={"company_symbol_2":"issuer"})
]).groupby("issuer").size().reset_index(name="n_useful_pairs").sort_values("n_useful_pairs", ascending=False)

df.sort_values("post_regime_rank_score", ascending=False).to_csv(os.path.join(outdir, "stage2_regime_results_useful_with_post_analysis.csv"), index=False)
presentation.to_csv(os.path.join(outdir, "stage2_regime_results_presentation_subset.csv"), index=False)
summary_counts.to_csv(os.path.join(outdir, "stage2_regime_post_analysis_report.csv"), index=False)
type_counts.to_csv(os.path.join(outdir, "stage2_regime_pair_type_counts.csv"), index=False)
sector_combo_summary.to_csv(os.path.join(outdir, "stage2_regime_sector_combo_summary_post.csv"), index=False)
comparison.to_csv(os.path.join(outdir, "stage2_regime_half_life_comparison.csv"), index=False)
issuer_usage.to_csv(os.path.join(outdir, "stage2_regime_issuer_usage_post.csv"), index=False)

with open(os.path.join(outdir, "stage2_regime_post_analysis_config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)

readme = """Stage 2 post-regime analysis outputs
===================================

Files:
1. stage2_regime_results_useful_with_post_analysis.csv
   Useful regime pairs with added classification columns and a post-regime ranking score.

2. stage2_regime_results_presentation_subset.csv
   Smaller thesis-ready subset chosen from the high-quality useful set with issuer and sector-combo caps.

3. stage2_regime_post_analysis_report.csv
   High-level counts for useful, high-quality useful, and presentation subsets.

4. stage2_regime_pair_type_counts.csv
   Counts by regime distinction type, reversion pattern, and stress driver type.

5. stage2_regime_sector_combo_summary_post.csv
   Sector-combo summary with high-quality counts and median regime diagnostics.

6. stage2_regime_half_life_comparison.csv
   Comparison of Stage 1 half-life with regime-specific half-lives.

7. stage2_regime_issuer_usage_post.csv
   Issuer frequency in the useful regime set.
"""
with open(os.path.join(outdir, "README_stage2_regime_post_analysis.txt"), "w") as f:
    f.write(readme)

if CONFIG["zip_outputs"]:
    zip_path = os.path.join(outdir, f"{CONFIG['zip_basename']}.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fname in os.listdir(outdir):
            full = os.path.join(outdir, fname)
            if os.path.isfile(full) and fname != os.path.basename(zip_path):
                zf.write(full, arcname=fname)
    maybe_colab_download(zip_path)

print("Done")
summary_counts

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done


,metric,value
0,useful_pairs_in,442
1,high_quality_useful_pairs,252
2,presentation_pairs_selected,96
3,unique_sector_combos_useful,54
4,unique_sector_combos_presentation,38
5,unique_issuers_useful,65
6,unique_issuers_presentation,59
